# Planning section
Includes fields to keep/ignore for each data source (NUS, NTU, SUTD courses, and job descriptions), and the target schema for courses and jobs:

## For NUS:
    keep:
        "department"
        "description"
        "faculty"
        "moduleCode"
        "title"
        
    ignore:
        "gradingBasisDescription"
        "moduleCredit"
        "semesterData"
        "workload"


## For NTU:
    keep:
        "code" 
        "title" 
        "description" 
        "departmentMaintaining"

    ignore:
        "academicUnits"
        "examSchedule" 
        "gradeType" 
        "prerequisites" 
        "notAvailableToProgramme" (eg not available to EEE)
        "url" 
        "details" 


## For SUTD:
    keep:
        "code"
        "title" 
        "course_type" (core, electives, etc)
        "description" 
        "affiliations" (department)

    ignore:
        "terms" 
        "credits" 
        "url" 
        "description_found" (T/F)


extract skills from the descriptions

# courses:
|university| department| faculty| code| title| course_type| description| skills|

# jobs:
## keep: 
        "title"
        "description"
        "minimumYearsExperience"
        "shiftPattern"
        "skills" (keep everything in the skills section)
            "uuid": "11ff9f68afb6b8b5b8eda218d7c83a65",
            "confidence": null,
            "isKeySkill": null
        "otherRequirements"
        "ssocCode" (job industry/sector)
        "occupationId"
        "ssocVersion"
        "workingHours"
        "numberOfVacancies"
        "categories"
            "id"
            "category"
        "employmentTypes" 
            "id" 
            "employmentType" 
        "positionLevels"
            "id": 11,
            "position" 
        "ssecEqa" 
            (Singapore Standard Educational Classification (SSEC) Educational Qualification Assessment code, indicating the education level. 
            eg "69" typically corresponds to a Bachelor's Degree with Honours.)
        "ssecFos" 
            (Field of Study code, 
            eg "0212" corresponds to "Music and Performing Arts")
        "ssicCode" 
            (It classifies the type of industry/business sector the company operates in.)
            (SSOC = the job classification (Laboratory Technician, Finance Manager, etc.)
            (SSIC = the company/industry classification (Manufacturing, Education, etc.))
        "ssicCode2020" 
         "salary": {
                    "maximum": 3500,
                    "minimum": 2500,
                    "type": {
                    "id": 4,
                    "salaryType": "Monthly"
                    }
        "jobTitles"

## keep but tbc: (if all NA/nulls -> drop)
        "schemes" 
        "flexibleWorkArrangements"

## ignore:
        "uuid" (post id? user id?)
        "sourceCode" 
        "psdUrl"
        "status"
        "postedCompany" 
            "uen" 
                ("uen" stands for Unique Entity Number, a unique identifier assigned to all business entities (companies, organizations, partnerships) registered in Singapore. In the example, "52875677W" is the UEN for SYMPHONY MUSIC SCHOOL. It's used to uniquely identify and track companies in Singapore's business registry.)
            "description" (company description)
            "name" (company name)
        "employeeCount" 
        "companyUrl"
        "lastSyncDate": "2026-01-30T02:57:44.000Z",
        "_links"
        "logoFileName" 
        "logoUploadPath" 
        "responsiveEmployer": {
      "isResponsive": false
    }



# NUS

## Load Data

In [218]:
# import necessary libraries
import json5
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# load output NUSModsInfo.json file
nus_file_path = Path("../../data/NUSModsInfo.json")

with open(nus_file_path, "r", encoding="utf-8") as f:
    nus_data = json5.load(f)

In [ ]:
# Extract relevant fields
nus_fields = ["department", "description", "faculty", "moduleCode", "title"]

nus_filtered_data = [
    {key: item.get(key) for key in nus_fields}
    for item in nus_data
]

In [ ]:
# Create DataFrame
nus_df = pd.DataFrame(nus_filtered_data)

# Preview
nus_df.head()

,department,description,faculty,moduleCode,title
0,NUS Medicine Dean's Office,Leadership is fundamental to the success of in...,Yong Loo Lin Sch of Medicine,ABM5001,Leadership in Biomedicine
1,NUS Medicine Dean's Office,This course serves as a concept-based introduc...,Yong Loo Lin Sch of Medicine,ABM5002,Advanced Biostatistics for Research
2,NUS Medicine Dean's Office,This course will furnish students with a thoro...,Yong Loo Lin Sch of Medicine,ABM5003,Biomedical Innovation & Enterprise
3,NUS Medicine Dean's Office,This course encompasses research projects rele...,Yong Loo Lin Sch of Medicine,ABM5004,Capstone Project
4,NUS Medicine Dean's Office,Advanced immunological applications play impor...,Yong Loo Lin Sch of Medicine,ABM5101,Applied Immunology


## Data Cleaning

### Data Exploration

In [ ]:
# check for missing values
print(nus_df.isnull().sum())

department     103
description      0
faculty          0
moduleCode       0
title            0
dtype: int64


In [ ]:
nus_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20514 entries, 0 to 20513
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   department   20411 non-null  object
 1   description  20514 non-null  object
 2   faculty      20514 non-null  object
 3   moduleCode   20514 non-null  object
 4   title        20514 non-null  object
dtypes: object(5)
memory usage: 801.5+ KB


In [ ]:
# unique values in department and faculty
print("Unique departments:", nus_df["department"].nunique())
print("Unique faculties:", nus_df["faculty"].nunique()) 

Unique departments: 109
Unique faculties: 25


In [ ]:
# get unique faculties 
unique_faculties = nus_df["faculty"].unique()
print("Unique faculties:", unique_faculties)

Unique faculties: ['Yong Loo Lin Sch of Medicine' 'College of Design and Engineering'
 'NUS Business School' 'Arts and Social Science' 'NUS' 'Computing' 'Law'
 'Cont and Lifelong Education' 'Science' 'Non-Faculty-based Departments'
 'Dentistry' 'YST Conservatory of Music' 'Multi Disciplinary Programme'
 'NUS-ISS' 'Residential College' 'Temasek Defence Sys. Institute'
 'Risk Management Institute' 'NUS College' 'SSH School of Public Health'
 'Duke-NUS Medical School' 'NUS Graduate School' 'Logistics Inst-Asia Pac'
 'Mechanobiology Institute (MBI)' 'LKY School of Public Policy'
 'Yale-NUS College']


In [ ]:
# faculty level data distribution
faculty_counts = nus_df["faculty"].value_counts()
print(faculty_counts)

faculty
Arts and Social Science              5677
Law                                  2878
College of Design and Engineering    2823
Science                              1695
NUS Business School                  1642
Yale-NUS College                     1485
Computing                             541
Yong Loo Lin Sch of Medicine          517
Cont and Lifelong Education           425
Duke-NUS Medical School               416
LKY School of Public Policy           376
YST Conservatory of Music             373
Non-Faculty-based Departments         366
NUS College                           337
Residential College                   261
SSH School of Public Health           208
NUS                                   154
NUS-ISS                                82
NUS Graduate School                    65
Dentistry                              59
Temasek Defence Sys. Institute         51
Risk Management Institute              48
Multi Disciplinary Programme           15
Mechanobiology Institute (

### Filter for undergraduate courses only

In [ ]:
# Keep only undergraduate level courses: codes where the first digit after letters is 0-4
nus_df = nus_df[nus_df['moduleCode'].str.match(r'^[A-Z]+[0-4]')]

nus_df

,department,description,faculty,moduleCode,title
27,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701,Accounting for Decision Makers
28,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701A,Accounting for Decision Makers
29,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701B,Accounting for Decision Makers
30,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701C,Accounting for Decision Makers
31,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701D,Accounting for Decision Makers
...,...,...,...,...,...
20509,Biological Sciences,In addition to having an academic science foun...,Science,ZB3312,Enhanced Undergraduate Professional Internship...
20510,Biological Sciences,In addition to having an academic science foun...,Science,ZB3313,Undergraduate Professional Internship Programm...
20511,Biological Sciences,This is a seminar-style course based on the li...,Science,ZB4171,Advanced Topics in Bioinformatics
20512,Biological Sciences,Not Available,Science,ZB4199,Honours Project in Computational Biology


In [ ]:
# faculty level data distribution
faculty_counts = nus_df["faculty"].value_counts()
print(faculty_counts)

faculty
Arts and Social Science              4629
Yale-NUS College                     1485
College of Design and Engineering    1374
Science                              1172
NUS Business School                   796
Law                                   783
Cont and Lifelong Education           403
Computing                             350
YST Conservatory of Music             337
NUS College                           337
Non-Faculty-based Departments         333
Residential College                   261
Yong Loo Lin Sch of Medicine          174
NUS                                   122
SSH School of Public Health            61
Dentistry                              50
NUS-ISS                                19
Multi Disciplinary Programme           15
Duke-NUS Medical School                 3
Temasek Defence Sys. Institute          2
Name: count, dtype: int64


In [ ]:
# filter out post graduate level courses: faculty is for post graduate level courses, which are not relevant to our analysis
target_faculties = [
    "Temasek Defence Sys. Institute",
     "Cont and Lifelong Education",
     "Duke-NUS Medical School"
]

filtered_nus_df = nus_df[~nus_df["faculty"].isin(target_faculties)]

### Handling NA Values

In [ ]:
# count data with null description
null_description_count = filtered_nus_df["description"].isnull().sum()
print("Number of courses with null description:", null_description_count)

Number of courses with null description: 0


In [ ]:
# Distribution of description data
description_counts = filtered_nus_df["description"].value_counts()
print(description_counts)

description
Not Available                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           3663
UTOP aims to train undergraduates to acquire an

some short descriptions such as: NIL, Exchange Course - YUS (1 unit), Not Applicable 
are not meaningful for analysis. we would remove those data

In [ ]:
# Keep only rows where description has 10 or more words
filtered_nus_df = filtered_nus_df[filtered_nus_df['description'].str.split().str.len() >= 10]

In [ ]:
filtered_nus_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8499 entries, 27 to 20513
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   department   8499 non-null   object
 1   description  8499 non-null   object
 2   faculty      8499 non-null   object
 3   moduleCode   8499 non-null   object
 4   title        8499 non-null   object
dtypes: object(5)
memory usage: 398.4+ KB


there is no more NA data in the dataset

## Data Transformation

In [ ]:
filtered_nus_df["University"] = "NUS"

In [ ]:
filtered_nus_df.head()

,department,description,faculty,moduleCode,title,University
27,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701,Accounting for Decision Makers,NUS
28,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701A,Accounting for Decision Makers,NUS
29,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701B,Accounting for Decision Makers,NUS
30,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701C,Accounting for Decision Makers,NUS
31,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701D,Accounting for Decision Makers,NUS


# NTU

## Load Data

In [ ]:
# load output ntuCourseInfo.json file
ntu_file_path = Path("../../data/ntuCourseInfo.json")

with open(ntu_file_path, "r", encoding="utf-8") as f:
    ntu_data = json5.load(f)

In [ ]:
# Extract relevant fields
ntu_fields =  ["code", "title", "description", "departmentMaintaining"]

ntu_filtered_data = [
    {key: item.get(key) for key in ntu_fields}
    for item in ntu_data
]

In [ ]:
# Create DataFrame
ntu_df = pd.DataFrame(ntu_filtered_data)

# Preview
ntu_df.head()

,code,title,description,departmentMaintaining
0,AAA08B,AAA08B FASHION & DESIGN: WEARABLE ART AS A SEC...,Develop process skills in expression and visua...,NIE
1,AAA08C,AAA08C EXPRESSIVE DRAWING: DEVELOPING PERSONAL...,Students will learn a brief history of express...,NIE
2,AAA08D,AAA08D ABSTRACT PAINTING: WHY IT'S HERE & HOW ...,Students will learn a brief history of abstrac...,NIE
3,AAA18D,AAA18D LIFE DRAWING,This course introduces drawing of the nude hum...,NIE
4,AAA18E,AAA18E DRAWING,This course introduces drawing at a basic leve...,NIE


## Data Cleaning

### Data Exploration

In [ ]:
# check for missing values
print(ntu_df.isnull().sum())

code                      0
title                     0
description               0
departmentMaintaining    14
dtype: int64


In [ ]:
ntu_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2117 entries, 0 to 2116
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   code                   2117 non-null   object
 1   title                  2117 non-null   object
 2   description            2117 non-null   object
 3   departmentMaintaining  2103 non-null   object
dtypes: object(4)
memory usage: 66.3+ KB


In [ ]:
# unique values in department
print("Unique departments:", ntu_df["departmentMaintaining"].nunique())

Unique departments: 47


In [ ]:
# faculty level data distribution
ntu_dept_counts = ntu_df["departmentMaintaining"].value_counts()
print(ntu_dept_counts)

departmentMaintaining
BUS          144
ADM          144
CSC(CE)      129
NIE          119
SOH          111
EEE           96
CS            96
ELH(SOH)      74
MATH(SPS)     71
CHIN(SOH)     67
ME            64
BS            62
LMS(SOH)      57
PSY(SSS)      56
EESS(ASE)     54
HIST(SOH)     53
MAT           53
ECON(SSS)     51
PHY(SPS)      49
CBE           49
PPGA(SSS)     49
PHIL(SOH)     48
SOC(SSS)      41
CEE           39
CHEM(CBE)     37
CE            36
ACC           36
BIE(CBE)      28
REP           25
SSM(NIE)      22
MS(CEE)       21
ENE(CEE)      20
AERO(ME)      20
CMED(BS)      15
NTC           14
BMS(BS)       13
SSS            9
ICC            8
SPS            6
IEM(EEE)       4
BSB(BS)        4
ROBO(ME)       3
BACF(CE)       2
CAO            1
BSB            1
ACDA(CE)       1
BMS            1
Name: count, dtype: int64


### Filter for undergraduate courses only

In [ ]:
# Keep only undergraduate level courses: codes where the first digit after letters is 0-4
ntu_df = ntu_df[ntu_df['code'].str.match(r'^[A-Z]+[0-4]')]

### Add department name

In [ ]:
# import openpyxl if you can't load
# import openpyxl

In [ ]:
ntu_dept_file_path = Path("../../data/ntu_dept_mapping.xlsx")

ntu_dept_df = pd.read_excel(ntu_dept_file_path)

ntu_dept_df.head()


,short,department
0,ACC,Nanyang Business School (Accountancy)
1,ACDA(CE),Programme under Civil Engineering
2,ADM,"School of Art, Design and Media ..."
3,AERO(ME),Aerospace Engineering (MAE)
4,BACF(CE),Bachelor of Applied Computing in Finance (CE)


In [ ]:
# add department mapping to ntu_df
ntu_df = ntu_df.merge(
    ntu_dept_df[["short", "department"]],
    left_on="departmentMaintaining",
    right_on="short",
    how="left"
).drop(columns=["short"])

In [ ]:
ntu_df.rename(columns={"departmentMaintaining": "dept_code"}, inplace=True)

### Handling NA Values

In [ ]:
# count data with null description
ntu_null_description_count = ntu_df["description"].isna().sum()
print("Number of courses with null description:", ntu_null_description_count)

Number of courses with null description: 0


In [ ]:
# Distribution of description data
ntu_description_counts = ntu_df["description"].value_counts()
print(ntu_description_counts)

description
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [ ]:
# Keep only rows where description has 10 or more words
filtered_ntu_df = ntu_df[ntu_df['description'].str.split().str.len() >= 10]

In [ ]:
filtered_ntu_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1818 entries, 0 to 1851
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   code         1818 non-null   object
 1   title        1818 non-null   object
 2   description  1818 non-null   object
 3   dept_code    1818 non-null   object
 4   department   1818 non-null   object
dtypes: object(5)
memory usage: 85.2+ KB


## Data Transformation

In [ ]:
filtered_ntu_df["University"] = "NTU"

/var/folders/j6/fp83dr1s70x5cd1fz38zrdn80000gp/T/ipykernel_18530/858357989.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_ntu_df["University"] = "NTU"


In [ ]:
filtered_ntu_df.head()

,code,title,description,dept_code,department,University
0,AAA08B,AAA08B FASHION & DESIGN: WEARABLE ART AS A SEC...,Develop process skills in expression and visua...,NIE,National Institute of Education,NTU
1,AAA08C,AAA08C EXPRESSIVE DRAWING: DEVELOPING PERSONAL...,Students will learn a brief history of express...,NIE,National Institute of Education,NTU
2,AAA08D,AAA08D ABSTRACT PAINTING: WHY IT'S HERE & HOW ...,Students will learn a brief history of abstrac...,NIE,National Institute of Education,NTU
3,AAA18D,AAA18D LIFE DRAWING,This course introduces drawing of the nude hum...,NIE,National Institute of Education,NTU
4,AAA18E,AAA18E DRAWING,This course introduces drawing at a basic leve...,NIE,National Institute of Education,NTU


# SUTD

## Load Data

In [ ]:
# load output from sutdCourseDescriptions.json file
sutd_file_path = Path("../../data/sutdCourseDescriptions.json")

with open(sutd_file_path, "r", encoding="utf-8") as f:
    sutd_data = json5.load(f)

In [ ]:
# Extract relevant fields
sutd_fields =  ["code", "title", "course_type", "description", "affiliations"]

sutd_filtered_data= [
    {key: item.get(key) for key in sutd_fields}
    for item in sutd_data
]

In [ ]:
# Create DataFrame
sutd_df = pd.DataFrame(sutd_filtered_data)

sutd_df.head()

,code,title,course_type,description,affiliations
0,01.018,Design Thinking Project I,Freshmore core,"Design Thinking Projects I, II, and III provid...",None
1,01.019,Design Thinking Project II,Freshmore core,"Design Thinking Projects I, II, and III provid...",None
2,01.020,Design Thinking Project III,Freshmore core,"Design Thinking Projects I, II, and III provid...",None
3,01.101,Technologies for Sustainable Global Health,Elective / Technical Elective,This course provides a multi-disciplinary appr...,None
4,01.102,Energy Systems and Management,Elective / Technical Elective,"Broadly, this course aims to acquaint students...",None


## Data Cleaning

### Data Exploration

In [ ]:
# check for missing values
print(sutd_df.isnull().sum())

code              0
title             0
course_type       0
description       5
affiliations    204
dtype: int64


### Add department name

In [ ]:
AFFILIATION_NAMES = {
    "ASD": "Architecture and Sustainable Design",
    "DAI": "Design and Artificial Intelligence",
    "EPD": "Engineering Product Development",
    "ESD": "Engineering Systems and Design",
    "HASS": "Humanities, Arts and Social Sciences",
    "ISTD": "Information Systems Technology and Design",
    "SMT": "Science, Mathematics and Technology",
}

sutd_df["department"] = sutd_df["affiliations"].apply(
    lambda codes: [AFFILIATION_NAMES[c] for c in (codes or []) if c in AFFILIATION_NAMES]
)

### Handling NA Values

In [ ]:
# count data with null description
sutd_null_description_count = sutd_df["description"].isna().sum()
print("Number of courses with null description:", sutd_null_description_count)

Number of courses with null description: 5


In [ ]:
# Distribution of description data
sutd_description_counts = sutd_df["description"].value_counts()
print(sutd_description_counts)

description
Design Thinking Projects I, II, and III provide the Freshmore students the opportunity to learn and practice fundamental design thinking principles. Students are introduced to design thinking through a series of design-centric, interdisciplinary, multidisciplinary, hands-on projects and seminars, guided by a yearly general theme to ensure integrated pedagogy and progressive learning. Moreover, the projects and their solutions are expected to impact areas of growth at SUTD, such as healthcare, cities, aviation, and data science. 01.018 Design Thinking Project I in term 1 is a guided project, 01.019 Design Thinking Projects II in term 2 is integrated with 3.007 Design Thinking and Innovation course, while 01.020 Design Thinking Project III in term 3 allows the students with more scope for self-guidance. Ultimately, the courses are designed to equip the students with a series of design thinking, technical, contextual, organizational, leadership, scientific and technological sk

In [ ]:
# remove courses with description less than 10 words and NA descriptions
filtered_sutd_df = sutd_df[sutd_df['description'].str.split().str.len() >= 10]

## Data Transformation

In [ ]:
filtered_sutd_df["University"] = "SUTD"

/var/folders/j6/fp83dr1s70x5cd1fz38zrdn80000gp/T/ipykernel_18530/1833491157.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_sutd_df["University"] = "SUTD"


In [ ]:
filtered_sutd_df.head()

,code,title,course_type,description,affiliations,department,University
0,01.018,Design Thinking Project I,Freshmore core,"Design Thinking Projects I, II, and III provid...",None,[],SUTD
1,01.019,Design Thinking Project II,Freshmore core,"Design Thinking Projects I, II, and III provid...",None,[],SUTD
2,01.020,Design Thinking Project III,Freshmore core,"Design Thinking Projects I, II, and III provid...",None,[],SUTD
3,01.101,Technologies for Sustainable Global Health,Elective / Technical Elective,This course provides a multi-disciplinary appr...,None,[],SUTD
4,01.102,Energy Systems and Management,Elective / Technical Elective,"Broadly, this course aims to acquaint students...",None,[],SUTD


# Jobs

## Load Data

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

def extract_job_jsons_to_df(job_folder="../../data") -> pd.DataFrame:
    job_folder = Path(job_folder)
    rows = []

    files = [f for f in job_folder.rglob("*.json") if f.name.startswith("MCF-")]
    total_files = len(files)

    for i, file_path in enumerate(files, start=1):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            salary = data.get("salary", {}) or {}
            salary_type = salary.get("type", {}) or {}

            row = {
                "file_name": file_path.name,
                "title": data.get("title"),
                "description": data.get("description"),
                "minimumYearsExperience": data.get("minimumYearsExperience"),
                "shiftPattern": data.get("shiftPattern"),
                "otherRequirements": data.get("otherRequirements"),
                "ssocCode": data.get("ssocCode"),
                "occupationId": data.get("occupationId"),
                "ssocVersion": data.get("ssocVersion"),
                "workingHours": data.get("workingHours"),
                "numberOfVacancies": data.get("numberOfVacancies"),
                "ssecEqa": data.get("ssecEqa"),
                "ssecFos": data.get("ssecFos"),
                "ssicCode": data.get("ssicCode"),
                "ssicCode2020": data.get("ssicCode2020"),
                "jobTitles": data.get("jobTitles"),
                "skills": data.get("skills"),
                "categories": data.get("categories"),
                "employmentTypes": data.get("employmentTypes"),
                "positionLevels": data.get("positionLevels"),
                "salary_minimum": salary.get("minimum"),
                "salary_maximum": salary.get("maximum"),
                "salary_type_id": salary_type.get("id"),
                "salary_type": salary_type.get("salaryType"),
            }

            rows.append(row)

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

        if i % 1000 == 0:
            print(f"Processed {i}/{total_files} files ({(i/total_files)*100:.1f}%)")

    print(f"Done. Total processed: {total_files}")

    return pd.DataFrame(rows)

In [ ]:
df_jobs = extract_job_jsons_to_df("../../data")
print(f"Loaded job records: {df_jobs.shape[0]}")
print(df_jobs.head())

Processed 1000/22718 files (4.4%)
Processed 2000/22718 files (8.8%)
Processed 3000/22718 files (13.2%)
Processed 4000/22718 files (17.6%)
Processed 5000/22718 files (22.0%)
Processed 6000/22718 files (26.4%)
Processed 7000/22718 files (30.8%)
Processed 8000/22718 files (35.2%)
Processed 9000/22718 files (39.6%)
Processed 10000/22718 files (44.0%)
Processed 11000/22718 files (48.4%)
Processed 12000/22718 files (52.8%)
Processed 13000/22718 files (57.2%)
Processed 14000/22718 files (61.6%)
Processed 15000/22718 files (66.0%)
Processed 16000/22718 files (70.4%)
Processed 17000/22718 files (74.8%)
Processed 18000/22718 files (79.2%)
Processed 19000/22718 files (83.6%)
Processed 20000/22718 files (88.0%)
Processed 21000/22718 files (92.4%)
Processed 22000/22718 files (96.8%)
Done. Total processed: 22718
Loaded job records: 22718
               file_name                                              title  \
0  MCF-2026-0180015.json  Outdoor Sales Engineer (Own Car) | Petrochemic...   
1  MCF

In [ ]:
df_jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22718 entries, 0 to 22717
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   file_name               22718 non-null  object
 1   title                   22718 non-null  object
 2   description             22718 non-null  object
 3   minimumYearsExperience  22718 non-null  int64 
 4   shiftPattern            0 non-null      object
 5   otherRequirements       0 non-null      object
 6   ssocCode                22718 non-null  object
 7   occupationId            22718 non-null  object
 8   ssocVersion             22718 non-null  object
 9   workingHours            0 non-null      object
 10  numberOfVacancies       22718 non-null  int64 
 11  ssecEqa                 22718 non-null  object
 12  ssecFos                 15444 non-null  object
 13  ssicCode                0 non-null      object
 14  ssicCode2020            0 non-null      object
 15  jo

# All Uni Courses Dataframe


|university| department| faculty| code| title| course_type| description| skills|

In [ ]:
# Standardize column names for consistency
# NUS: rename 'moduleCode' to 'code'
filtered_nus_df = filtered_nus_df.rename(columns={'moduleCode': 'code'})

# NTU: already has 'code', but 'dept_code' and 'department' are separate
# Keep 'department' as the full name, drop 'dept_code' if not needed

# SUTD: already has 'code', 'department' is a list, but we'll keep it

# Create unified columns: ['university', 'department', 'faculty', 'code', 'title', 'course_type', 'description']

# For NUS
nus_unified = filtered_nus_df[['University', 'department', 'faculty', 'code', 'title', 'description']].copy()
nus_unified['course_type'] = None  # NUS doesn't have course_type

# For NTU
ntu_unified = filtered_ntu_df[['University', 'department', 'code', 'title', 'description']].copy()
ntu_unified['faculty'] = None  # NTU doesn't have faculty
ntu_unified['course_type'] = None

# For SUTD
sutd_unified = filtered_sutd_df[['University', 'department', 'code', 'title', 'course_type', 'description']].copy()
sutd_unified['faculty'] = None  # SUTD doesn't have faculty

# Concatenate all courses
all_courses_df = pd.concat([nus_unified, ntu_unified, sutd_unified], ignore_index=True)

# Rename columns to match target schema
all_courses_df = all_courses_df.rename(columns={'University': 'university'})

print(f"Consolidated courses shape: {all_courses_df.shape}")
all_courses_df.head()

Consolidated courses shape: (10516, 7)


,university,department,faculty,code,title,description,course_type
0,NUS,Accounting,NUS Business School,ACC1701,Accounting for Decision Makers,The course provides an introduction to account...,None
1,NUS,Accounting,NUS Business School,ACC1701A,Accounting for Decision Makers,The course provides an introduction to account...,None
2,NUS,Accounting,NUS Business School,ACC1701B,Accounting for Decision Makers,The course provides an introduction to account...,None
3,NUS,Accounting,NUS Business School,ACC1701C,Accounting for Decision Makers,The course provides an introduction to account...,None
4,NUS,Accounting,NUS Business School,ACC1701D,Accounting for Decision Makers,The course provides an introduction to account...,None


# Text Preprocessing

In [ ]:
import re
from bs4 import BeautifulSoup

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Lowercase
    text = text.lower()
    return text

# Apply to courses
all_courses_df['description_clean'] = all_courses_df['description'].apply(clean_text)

# Apply to jobs (check if column exists)
if 'description' in df_jobs.columns:
    df_jobs['description_clean'] = df_jobs['description'].apply(clean_text)
else:
    print("Warning: 'description' column not found in df_jobs")
    df_jobs['description_clean'] = ""

print("Text preprocessing completed for courses and jobs.")

Text preprocessing completed for courses and jobs.


# Skills Extraction

In [ ]:
# pip install torch keybert sentence-transformers

In [ ]:
# Simple Skill Extraction from Job Data
print("\n=== Skill Extraction ===")

# Extract skills directly from job postings
def extract_skills_from_jobs(jobs_df):
    """Extract unique skills from job postings"""
    skills_vocab = set()
    if 'skills' in jobs_df.columns:
        for skills_list in jobs_df['skills'].dropna():
            if isinstance(skills_list, list):
                for item in skills_list:
                    if isinstance(item, dict) and 'skill' in item:
                        skill = str(item['skill']).strip()
                        if skill and len(skill) > 2:
                            skills_vocab.add(skill)
    return sorted(list(skills_vocab))

skill_vocab = extract_skills_from_jobs(df_jobs)
print(f"Extracted {len(skill_vocab)} unique skills from job data")

# Add skills column to courses (placeholder for now)
all_courses_df['skills'] = all_courses_df['description'].apply(lambda x: [])

def normalize_skills(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    # For array-likes (e.g. pandas Series) and iterables that are not strings/dicts
    if not isinstance(x, (str, bytes, dict)):
        try:
            iterable = list(x)
            return iterable
        except TypeError:
            pass
    # If skill entries are stored as strings, preserve as single-item list
    if isinstance(x, str) and x.strip():
        return [x.strip()]
    return []

# Use normalized skills list for extracted skills
if 'skills' in df_jobs.columns:
    df_jobs['skills_extracted'] = df_jobs['skills'].apply(normalize_skills)
else:
    df_jobs['skills_extracted'] = []

print(f"Skills columns added to courses and jobs")


=== Skill Extraction ===
Extracted 9456 unique skills from job data
Skills columns added to courses and jobs


# Job Data Cleaning

In [ ]:
print("\n=== Job Data Cleaning ===")

# Create description_clean if it doesn't exist
if 'description_clean' not in df_jobs.columns:
    print("Creating description_clean column from description...")
    df_jobs['description_clean'] = df_jobs['description'].apply(
        lambda x: clean_text(x) if isinstance(x, str) else ""
    )

# Filter for meaningful descriptions (>=10 words)
before_clean = len(df_jobs)
valid_desc = df_jobs['description_clean'].str.split().str.len() >= 10
df_jobs = df_jobs[valid_desc]
print(f"Filtered by description length: {before_clean} -> {len(df_jobs)} records")

# Remove null titles and descriptions
before_null = len(df_jobs)
df_jobs = df_jobs.dropna(subset=['title', 'description'])
print(f"Removed null values: {before_null} -> {len(df_jobs)} records")

# Remove duplicate job postings (by title and description)
before_dup = len(df_jobs)
df_jobs = df_jobs.drop_duplicates(subset=['title', 'description'], keep='first')
print(f"Removed duplicates: {before_dup} -> {len(df_jobs)} records")

print(f"\nFinal cleaned jobs shape: {df_jobs.shape}")
print(f"Columns: {list(df_jobs.columns[:10])}...")  # Show first 10 columns


=== Job Data Cleaning ===
Filtered by description length: 18970 -> 18970 records
Removed null values: 18970 -> 18970 records
Removed duplicates: 18970 -> 18970 records

Final cleaned jobs shape: (18970, 26)
Columns: ['file_name', 'title', 'description', 'minimumYearsExperience', 'shiftPattern', 'otherRequirements', 'ssocCode', 'occupationId', 'ssocVersion', 'workingHours']...


# Save Cleaned Data

In [ ]:
print("\\n=== Data Validation ===" )

# Validate Courses DataFrame
print("\\n** Courses Validation **")
print(f"Shape: {all_courses_df.shape}")
print(f"Null counts:\\n{all_courses_df.isnull().sum()}")
print(f"\\nData types:\\n{all_courses_df.dtypes}")
print(f"\\nUnique universities: {all_courses_df['university'].unique()}")

# Validate Jobs DataFrame
print("\\n** Jobs Validation **")
print(f"Shape: {df_jobs.shape}")
print(f"Null counts:\\n{df_jobs.isnull().sum()}")
print(f"\\nKey columns present: {set(['title', 'description', 'skills']).issubset(set(df_jobs.columns))}")

# Data quality checks
print("\\n** Data Quality Checks **")
courses_no_description = (all_courses_df['description'].isna() | (all_courses_df['description'] == '')).sum()
jobs_no_description = (df_jobs['description'].isna() | (df_jobs['description'] == '')).sum()
print(f"Courses without description: {courses_no_description}")
print(f"Jobs without description: {jobs_no_description}")

if courses_no_description > 0:
    print("Removing courses without descriptions...")
    all_courses_df = all_courses_df[all_courses_df['description'].notna() & (all_courses_df['description'] != '')]
    print(f"Courses after cleanup: {all_courses_df.shape[0]}")

if jobs_no_description > 0:
    print("Removing jobs without descriptions...")
    df_jobs = df_jobs[df_jobs['description'].notna() & (df_jobs['description'] != '')]
    print(f"Jobs after cleanup: {df_jobs.shape[0]}")

print("\\n✓ Validation complete")

\n=== Data Validation ===
\n** Courses Validation **
Shape: (10516, 9)
Null counts:\nuniversity               0
department               0
faculty               2017
code                     0
title                    0
description              0
course_type          10317
description_clean        0
skills                   0
dtype: int64
\nData types:\nuniversity           object
department           object
faculty              object
code                 object
title                object
description          object
course_type          object
description_clean    object
skills               object
dtype: object
\nUnique universities: ['NUS' 'NTU' 'SUTD']
\n** Jobs Validation **
Shape: (18970, 26)
Null counts:\nfile_name                     0
title                         0
description                   0
minimumYearsExperience        0
shiftPattern              18970
otherRequirements         18970
ssocCode                      0
occupationId                  0
ssocVersion          

In [ ]:
# Save cleaned courses data
output_dir = Path("../../data/cleaned")
output_dir.mkdir(exist_ok=True)

all_courses_df.to_csv(output_dir / "courses_cleaned.csv", index=False)
df_jobs.to_csv(output_dir / "jobs_cleaned.csv", index=False)

print(f"Saved cleaned data to {output_dir}")
print(f"Courses: {all_courses_df.shape[0]} records")
print(f"Jobs: {df_jobs.shape[0]} records")

Saved cleaned data to ../../data/cleaned
Courses: 10516 records
Jobs: 18970 records
